In [1]:
%load_ext autotime

import scarf
import matplotlib.pyplot as plt
import pandas as pd

scarf.__version__

'1.0.0'

time: 2.5 s (started: 2026-06-29 18:27:50 +02:00)


In [2]:
scarf.fetch_dataset("lecun_60K_mnist_images", save_path="scarf_datasets")

time: 30.9 s (started: 2026-06-29 18:27:53 +02:00)


In [3]:
reader = scarf.CrDirReader("scarf_datasets/lecun_60K_mnist_images")
writer = scarf.CrToZarr(
    reader,
    zarr_loc="scarf_datasets/lecun_60K_mnist_images/data.zarr",
    chunk_size=(2000, 1000),
)
writer.dump(batch_size=1000)

  0%|                                                                                                         …

time: 5.7 s (started: 2026-06-29 18:28:24 +02:00)


In [4]:
ds = scarf.DataStore(
    "scarf_datasets/lecun_60K_mnist_images/data.zarr",
    min_cells_per_feature=1000,
    min_features_per_cell=10,
    nthreads=4,
)
ds

(RNA) Computing nCells and dropOuts:   0%|                                                                    …

(RNA) Computing nCounts:   0%|                                                                                …

(RNA) Computing nFeatures:   0%|                                                                              …

DataStore has 60000 (60000) cells with 1 assays: RNA
   Cell metadata:
            'I', 'ids', 'names', 'RNA_nCounts', 'RNA_nFeatures', 
          
   RNA assay has 467 (784) features and following metadata:
            'I', 'ids', 'names', 'nCells', 'dropOuts', 
          

time: 797 ms (started: 2026-06-29 18:28:29 +02:00)


In [5]:
ds.cells.insert(
    "digit_label",
    [int(x.rsplit("_", 1)[-1]) - 1 for x in ds.cells.fetch_all("names")],
    overwrite=True,
)

time: 54.6 ms (started: 2026-06-29 18:28:30 +02:00)


In [6]:
# Set normalization method to a dummy function that returns unnormalized data
ds.RNA.normMethod = scarf.assay.norm_dummy

ds.make_graph(feat_key="I", k=31, dims=25, n_centroids=100, show_elbow_plot=True)

INFO: make_graph step 1/7: normalize expression matrix (computing)


Writing data to normed__I__I/data:   0%|                                                                      …

INFO: make_graph step 1/7: normalize expression matrix finished in 1.0s (computing)


INFO: make_graph step 2/7: normalization statistics (computing)


Calculating mean and std. dev. of norm. data:   0%|                                                           …

...

Smoothening KNN distances:   0%|                                                                              …

INFO: make_graph step 6/7: smooth KNN distances into graph finished in 13.1s (computing)


INFO: make_graph finished in 31.3s (6/7 steps, 31.2s in logged steps)


OSError: [Errno 28] No space left on device

time: 32.4 s (started: 2026-06-29 18:28:30 +02:00)


In [7]:
ds.run_umap(n_epochs=300, spread=1, min_dist=0.05, parallel=True)

KeyError: 'latest_reduction'

time: 79.8 ms (started: 2026-06-29 18:29:03 +02:00)


In [8]:
ds.run_clustering(n_clusters=20)

KeyError: 'latest_reduction'

time: 175 ms (started: 2026-06-29 18:29:03 +02:00)


In [9]:
ds.smart_label(
    to_relabel="RNA_cluster",
    base_label="digit_label",
    new_col_name="cluster_label",
)

KeyError: 'RNA_cluster does not exist in the metadata columns.'

time: 211 ms (started: 2026-06-29 18:29:03 +02:00)


In [10]:
ds.plot_layout(
    layout_key="RNA_UMAP",
    color_by=["digit_label", "cluster_label"],
    do_shading=True,
    shade_npixels=300,
    legend_onside=False,
    width=4,
    height=4,
    cmap="tab20",
)

KeyError: 'RNA_UMAP1 does not exist in the metadata columns.'

time: 198 ms (started: 2026-06-29 18:29:03 +02:00)

In [11]:
ds.plot_cluster_tree(
    cluster_key="cluster_label",
    fill_by_value="digit_label",
)

KeyError: 'cluster_label does not exist in the metadata columns.'

time: 406 ms (started: 2026-06-29 18:29:03 +02:00)


In [12]:
clusts = pd.Series(ds.cells.fetch_all("cluster_label"))
digits = pd.Series(ds.cells.fetch_all("digit_label"))

KeyError: 'cluster_label does not exist in the metadata columns.'

time: 51 ms (started: 2026-06-29 18:29:04 +02:00)


In [13]:
fig = plt.figure(figsize=(8, 2))
for n, i in enumerate(sorted(clusts.unique())):
    mean_map = ds.RNA.rawData[((clusts == i) & (digits == int(i[0]))).values].mean(
        axis=0
    )
    mean_map = mean_map.compute().reshape(28, 28)
    ax = fig.add_subplot(2, 10, n + 1)
    ax.imshow(mean_map, cmap="binary")
    ax.set_axis_off()
    ax.set_title(i, fontsize=10)
plt.tight_layout()
plt.show()

NameError: name 'clusts' is not defined

<Figure size 800x200 with 0 Axes>

time: 61.3 ms (started: 2026-06-29 18:29:04 +02:00)
